### Helper functions 

In [ ]:
from collections import deque
import csv
from datetime import datetime, timedelta
import io
import math
import os
import random


def _estimate_csv_row_count(source_path: str) -> int:
    """Estimate data rows from file size / average line length (no full scan)."""
    file_size = os.path.getsize(source_path)
    if file_size == 0:
        return 0

    with open(source_path, "rb") as source_file:
        sample = source_file.read(1 << 20)

    newline_count = sample.count(b"\n") or 1
    avg_line_length = len(sample) / newline_count
    return max(0, int(file_size / avg_line_length) - 1)


def _read_csv_tail_values(source_path: str, n_rows: int, header: list) -> list:
    """Read the last n_rows data lines by seeking from EOF."""
    if n_rows <= 0:
        return []

    with open(source_path, "rb") as source_file:
        source_file.seek(0, os.SEEK_END)
        file_size = source_file.tell()
        if file_size == 0:
            return []

        target_lines = n_rows + 2
        block_size = 1 << 20
        buffer = b""
        position = file_size

        while position > 0 and buffer.count(b"\n") < target_lines:
            read_size = min(block_size, position)
            position -= read_size
            source_file.seek(position)
            buffer = source_file.read(read_size) + buffer

    text = buffer.decode("utf-8-sig", errors="replace")
    lines = text.splitlines()
    if position > 0 and lines:
        lines = lines[1:]

    if lines and lines[0].startswith(header[0].lstrip("\ufeff")):
        lines = lines[1:]

    data_lines = lines[-n_rows:] if len(lines) >= n_rows else lines
    if not data_lines:
        return []

    reader = csv.reader(io.StringIO("\n".join(data_lines) + "\n"))
    return [row for row in reader if row]


def inspect_source_csv_by_date(
    source_path: str, update_fraction: float, datetime_builder
):
    """
    Build an incremental source window without scanning multi-GB CSVs twice.

    Root cause of the hang:
    - hourly_88101_2024.csv is ~2.2GB / ~8M rows
    - old code did two full DictReader passes with strptime on every row
    - then printed every generated row

    Fix:
    - estimate row count from a size sample
    - seek from EOF to load a small tail
    - keep only rows inside the last (8760 * update_fraction) hours
    """
    with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
        header_reader = csv.reader(source_file)
        source_header = next(header_reader)

    source_row_count = _estimate_csv_row_count(source_path)
    # Read a slightly larger EOF chunk so hour-filtering still has enough rows
    target_size = max(1, math.ceil(source_row_count * update_fraction))
    read_size = max(target_size, math.ceil(source_row_count * min(0.05, update_fraction * 5)))

    tail_values = _read_csv_tail_values(source_path, read_size, source_header)
    source_tail = [dict(zip(source_header, values)) for values in tail_values]

    if not source_tail:
        buffer = deque(maxlen=read_size)
        with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
            reader = csv.reader(source_file)
            next(reader)
            counted = 0
            for values in reader:
                counted += 1
                buffer.append(values)
        source_row_count = counted
        source_tail = [dict(zip(source_header, values)) for values in buffer]

    dated_rows = []
    for row in source_tail:
        row_dt = datetime_builder(row)
        if row_dt is not None:
            dated_rows.append((row_dt, row))

    if not dated_rows:
        return source_header, source_row_count, [], None, None

    latest_dt = max(dt for dt, _ in dated_rows)
    target_hours = max(1, math.ceil(8760 * update_fraction))
    cutoff_dt = latest_dt - timedelta(hours=target_hours)
    source_tail = [row for dt, row in dated_rows if dt > cutoff_dt]

    if not source_tail:
        # Degenerate tail: keep rows at the latest timestamp
        source_tail = [row for dt, row in dated_rows if dt == latest_dt]
        cutoff_dt = latest_dt

    return source_header, source_row_count, source_tail, latest_dt, cutoff_dt


def write_incremental_csv(update_path: str, update_header: list, rows: list):
    os.makedirs(os.path.dirname(update_path), exist_ok=True)

    with open(update_path, "w", newline="", encoding="utf-8") as update_file:
        update_writer = csv.DictWriter(update_file, fieldnames=update_header)
        update_writer.writeheader()
        update_writer.writerows(rows)


def generate_incremental_update(
    source_path: str,
    update_path: str,
    update_fraction: float,
    datetime_builder,
    datetime_formatter,
    transform_row=None,
    extra_columns: list = None,
    time_step: timedelta = timedelta(hours=1),
    duplicate_fraction: float = 0.0,
):
    (
        source_header,
        source_row_count,
        source_tail,
        latest_dt,
        cutoff_dt,
    ) = inspect_source_csv_by_date(source_path, update_fraction, datetime_builder)

    if not source_tail or latest_dt is None:
        raise ValueError(f"No valid records or timestamps found in {source_path}")

    target_start_dt = latest_dt + time_step
    time_shift = target_start_dt - cutoff_dt - time_step

    generated_rows = []
    for source_row in source_tail:
        row_dict = dict(source_row)
        orig_dt = datetime_builder(row_dict)

        if orig_dt is not None:
            new_dt = orig_dt + time_shift
            row_dict = datetime_formatter(row_dict, new_dt, time_shift)

        if transform_row is not None:
            row_dict = transform_row(row_dict)

        generated_rows.append(row_dict)

    num_duplicates = math.ceil(len(generated_rows) * duplicate_fraction)
    duplicate_rows = (
        random.sample(source_tail, min(num_duplicates, len(source_tail)))
        if num_duplicates > 0 and source_tail
        else []
    )

    all_output_rows = generated_rows + [dict(r) for r in duplicate_rows]

    update_header = list(source_header) + list(extra_columns or [])
    write_incremental_csv(update_path, update_header, all_output_rows)

    period_start = datetime_builder(generated_rows[0])
    period_end = datetime_builder(generated_rows[-1])

    return (
        source_row_count,
        len(generated_rows),
        num_duplicates,
        period_start,
        period_end,
    )


### Air quality

In [8]:
from datetime import datetime

aqi_breakpoints = [
    (0.0, 9.0, 0, 50),
    (9.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 125.4, 151, 200),
    (125.5, 225.4, 201, 300),
    (225.5, 500.4, 301, 500),
]

print("Started")


def pm25_to_aqi(measurement):
    if measurement is None or str(measurement).strip() == "":
        return ""
    try:
        concentration = float(measurement)
        # EPA standard: clamp negative noise/sensor drift to 0.0
        concentration = max(0.0, concentration)
    except (TypeError, ValueError):
        return ""

    for concentration_low, concentration_high, aqi_low, aqi_high in aqi_breakpoints:
        if concentration <= concentration_high:
            aqi = ((aqi_high - aqi_low) / (concentration_high - concentration_low)) * (
                concentration - concentration_low
            ) + aqi_low
            return str(round(aqi))

    return "0"


def air_quality_dt_builder(row):
    try:
        dt_str = f"{row['Date Local']} {row['Time Local']}"
        return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
    except (KeyError, ValueError):
        return None


def air_quality_dt_formatter(row, new_dt, time_shift):
    row["Date Local"] = new_dt.strftime("%Y-%m-%d")
    row["Time Local"] = new_dt.strftime("%H:%M")

    if "Date GMT" in row and "Time GMT" in row and row["Date GMT"]:
        try:
            orig_gmt = datetime.strptime(
                f"{row['Date GMT']} {row['Time GMT']}", "%Y-%m-%d %H:%M"
            )
            new_gmt = orig_gmt + time_shift
            row["Date GMT"] = new_gmt.strftime("%Y-%m-%d")
            row["Time GMT"] = new_gmt.strftime("%H:%M")
        except ValueError:
            pass

    return row


def air_quality_transform(row):
    row["aqi"] = pm25_to_aqi(row.get("Sample Measurement", ""))
    return row


source_path = "../data/raw/air_quality/hourly_88101_2024.csv"
update_path = "../data/raw/air_quality/hourly_88101_update.csv"

print("Generating incremental update...")

source_row_count, new_records, duplicates, start_period, end_period = (
    generate_incremental_update(
        source_path=source_path,
        update_path=update_path,
        update_fraction=0.01,
        datetime_builder=air_quality_dt_builder,
        datetime_formatter=air_quality_dt_formatter,
        transform_row=air_quality_transform,
        extra_columns=["aqi"],
        duplicate_fraction=0.01,
    )
)

print("=== DATASET GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Duplicates injected:    {duplicates:,}")
print(f"Total update records:   {(new_records + duplicates):,}")
print(f"Schema Evolution:       Added column 'aqi' (Air Quality Index)")
print(f"Time period covered:    {start_period} to {end_period}")

Started
Generating incremental update...


KeyboardInterrupt: 

### Weather

In [3]:
from datetime import datetime, timedelta

def weather_dt_builder(row: dict) -> datetime:
    try:
        y = int(row["year"])
        m = int(row["month"])
        d = int(row["day"])
        h = int(float(row["hour"]))
        return datetime(y, m, d, h)
    except (KeyError, ValueError, TypeError):
        return None


def weather_dt_formatter(row: dict, new_dt: datetime, time_shift: timedelta) -> dict:
    row["year"] = str(new_dt.year)
    row["month"] = str(new_dt.month)
    row["day"] = str(new_dt.day)
    row["hour"] = str(new_dt.hour)
    return row


def weather_transform(row: dict) -> dict:
    rhum_val = row.get("rhum")
    if rhum_val not in (None, "", "NULL"):
        row["humidity"] = str(rhum_val)
    else:
        row["humidity"] = "60.0"
    return row


source_path = "../data/raw/weather/weather.csv"
update_path = "../data/raw/weather/weather_update.csv"

source_row_count, new_records, duplicates, start_period, end_period = (
    generate_incremental_update(
        source_path=source_path,
        update_path=update_path,
        update_fraction=0.01,
        datetime_builder=weather_dt_builder,
        datetime_formatter=weather_dt_formatter,
        transform_row=weather_transform,
        extra_columns=["humidity"],
        duplicate_fraction=0.01,
    )
)

print("=== DATASET GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Duplicates injected:    {duplicates:,}")
print(f"Total update records:   {(new_records + duplicates):,}")
print(f"Schema Evolution:       Added column 'humidity' (Relative Humidity %)")
print(f"Time period covered:    {start_period} to {end_period}")

=== DATASET GENERATION REPORT ===
Source file:            ../data/raw/weather/weather.csv
Source records:         8,784
Output update file:     ../data/raw/weather/weather_update.csv
New records written:    88
Duplicates injected:    1
Total update records:   89
Schema Evolution:       Added column 'humidity' (Relative Humidity %)
Time period covered:    2025-01-01 00:00:00 to 2025-01-04 15:00:00
